# MPI Implementation in CUDA (Python) — Distributed 2D Anisotropic Reaction-Diffusion Solver

**Student:** Ainat Yermurat  **ID:** 230103161  **GitHub:** [`yermy-bc`](https://github.com/yermy-bc)
**Instructor:** Sufyan bin Uzayr

4 MPI ranks (2 × 2 non-periodic Cartesian grid) × rank-local `258 × 258` subdomains on one Tesla T4,
GPU halo pack/unpack kernels, page-locked staging buffers, an `18 × 18` shared-memory tiled stencil and
a dual-stream (`stream_boundary` / `stream_interior`) compute–communication pipeline.

> Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

## Cell 1 — Environment verification & mpi4py installation

In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!which mpirun > /dev/null || (apt-get -qq update && apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null)
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-abeebd7d-e34b-f585-83bc-9af29ec895f6, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



## Cell 2 — Student identity configuration

In [2]:
# Replace with your exact GitHub username before running the solver
GITHUB_USERNAME = "yermy-bc"
STUDENT_NAME = "Ainat Yermurat"
STUDENT_ID = "230103161"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "You must set GITHUB_USERNAME to your real GitHub handle."
)
import os
os.environ["GITHUB_USERNAME"] = GITHUB_USERNAME
os.environ["STUDENT_NAME"] = STUDENT_NAME
os.environ["STUDENT_ID"] = STUDENT_ID
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")
print(f"Student: {STUDENT_NAME} ({STUDENT_ID})")

Configured student GitHub identity: yermy-bc
Student: Ainat Yermurat (230103161)


## Cell 3 — Distributed MPI + Numba-CUDA solver (`%%writefile distributed_cuda_solver.py`)

In [3]:
%%writefile distributed_cuda_solver.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
distributed_cuda_solver.py
==========================
4-rank distributed 2D anisotropic (heterogeneous-kappa) reaction-diffusion solver
built with mpi4py (distributed memory) + numba.cuda (SIMT GPU kernels).

Student : Ainat Yermurat (ID 230103161)
GitHub  : yermy-bc
Course  : MPI Implementation in CUDA (Python) - Instructor: Sufyan bin Uzayr

Implemented tasks
-----------------
Task 1  init_distributed_context   - 2x2 non-periodic Cartesian topology, GPU binding,
                                     SHA-256 hardware-bound INSTANCE_SEED
Task 2  initialize_subdomains      - rank-local (258,258) kappa / u0 / S fields,
                                     seeded Gaussian pulses, Dirichlet on PROC_NULL edges
Task 3  pack_halos_kernel / unpack_halos_kernel / exchange_halos_async
                                   - GPU strided packing, pinned host staging,
                                     tag-matched Irecv/Isend on the Cartesian communicator
Task 4  tiled_stencil_kernel / step_overlapped
                                   - 16x16 blocks, 18x18 shared tile + syncthreads,
                                     dual-stream compute/communication overlap
Task 5  run_solver                 - 400 steps serialized vs overlapped, Allreduce
                                     energy / L2 diagnostics, parity, PTX audit, manifest

Launch:  mpirun -np 4 python distributed_cuda_solver.py
(GITHUB_USERNAME must be exported in the environment.)
"""

import os

# Silence numba low-occupancy warnings (thin boundary-ring launches are intentional).
os.environ.setdefault("NUMBA_CUDA_LOW_OCCUPANCY_WARNINGS", "0")

import sys
import re
import json
import math
import time
import hashlib
import platform
import datetime
import subprocess
import warnings

import numpy as np
from mpi4py import MPI
import numba
from numba import cuda

warnings.filterwarnings("ignore", message=".*[Uu]nder-utilization.*")
warnings.filterwarnings("ignore", message=".*deprecat.*")
try:
    from numba.core.errors import NumbaPerformanceWarning, NumbaDeprecationWarning
    warnings.simplefilter("ignore", NumbaPerformanceWarning)
    warnings.simplefilter("ignore", NumbaDeprecationWarning)
except Exception:  # pragma: no cover
    pass

# ----------------------------------------------------------------------------
# Global problem constants
# ----------------------------------------------------------------------------
STUDENT_NAME = os.environ.get("STUDENT_NAME", "Ainat Yermurat")
STUDENT_ID = os.environ.get("STUDENT_ID", "230103161")

NY_GLOBAL, NX_GLOBAL = 512, 512          # interior global grid (no physical boundary)
PROC_DIMS = [2, 2]                        # 2 x 2 Cartesian process grid
PERIODS = [False, False]                  # non-periodic
NY, NX = NY_GLOBAL // PROC_DIMS[0], NX_GLOBAL // PROC_DIMS[1]   # 256 x 256 owned
LOCAL_SHAPE = (NY + 2, NX + 2)            # 258 x 258 incl. 1-cell ghost apron

DX = 1.0 / (NX_GLOBAL + 1)                # 1/513 (boundaries sit at x=0 and x=1)
DY = 1.0 / (NY_GLOBAL + 1)                # 1/513
LAMBDA = 0.15                             # cubic damping coefficient
KAPPA_LO, KAPPA_HI = 0.08, 0.24           # diffusivity range
CFL_SAFETY = 0.80                         # dt = 0.8 * dt_max
N_STEPS = 400
DIAG_EVERY = 100
TILE = 16                                 # 16 x 16 threadblock
TILE_HALO = 18                            # 18 x 18 shared tile (TILE + 2), literal for numba-cuda
PACK_THREADS = 128
PARITY_TOL = 1.0e-12
assert TILE_HALO == TILE + 2

# Message tags: tag names the direction the message TRAVELS.
TAG_NORTHWARD = 101   # my top owned row  -> neighbour above  (arrives in its bottom ghost)
TAG_SOUTHWARD = 102   # my bottom owned row -> neighbour below (arrives in its top ghost)
TAG_WESTWARD = 103    # my left owned col  -> neighbour left   (arrives in its right ghost)
TAG_EASTWARD = 104    # my right owned col -> neighbour right  (arrives in its left ghost)

# Layout of the packed 1-D halo vectors (length 2*NX + 2*NY):
#   [0      : NX)          top    strip  (send: row 1   | recv: ghost row 0)
#   [NX     : 2NX)         bottom strip  (send: row NY  | recv: ghost row NY+1)
#   [2NX    : 2NX+NY)      left   strip  (send: col 1   | recv: ghost col 0)
#   [2NX+NY : 2NX+2NY)     right  strip  (send: col NX  | recv: ghost col NX+1)
OFF_TOP, OFF_BOT, OFF_LEFT, OFF_RIGHT = 0, NX, 2 * NX, 2 * NX + NY
HALO_LEN = 2 * NX + 2 * NY


class Ctx:
    """Plain container for the distributed context."""
    pass


def root_print(ctx, *args):
    if ctx.rank == 0:
        print(*args, flush=True)


def ordered_print(ctx, msg):
    for r in range(ctx.size):
        if r == ctx.rank:
            print(msg, flush=True)
        ctx.cart.Barrier()


# ============================================================================
# TASK 1 - Distributed topology & hardware-bound seed
# ============================================================================
def _query_gpu_uuid(dev):
    """Return the active GPU UUID (e.g. 'GPU-1a2b...'). Never returns a placeholder."""
    uuid = None
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader", "-i", "0"],
            capture_output=True, text=True, timeout=30,
        )
        cand = out.stdout.strip().splitlines()
        if out.returncode == 0 and cand and cand[0].strip().startswith("GPU-"):
            uuid = cand[0].strip()
    except Exception:
        uuid = None
    if uuid is None:
        try:
            u = getattr(dev, "uuid", None)
            if u:
                u = u.decode() if isinstance(u, bytes) else str(u)
                uuid = u if u.startswith("GPU-") else "GPU-" + u
        except Exception:
            uuid = None
    if not uuid:
        raise RuntimeError("Could not determine the active GPU UUID - refusing to use a placeholder.")
    return uuid


def init_distributed_context(github_username):
    if (not github_username or not github_username.strip()
            or github_username.strip() == "REPLACE_WITH_YOUR_GITHUB_USERNAME"):
        raise RuntimeError("GITHUB_USERNAME is not set to a real GitHub handle.")
    github_username = github_username.strip()

    comm = MPI.COMM_WORLD
    size = comm.Get_size()
    assert size == 4, f"This solver requires exactly 4 MPI ranks, got {size}"

    cart = comm.Create_cart(dims=PROC_DIMS, periods=PERIODS, reorder=False)
    rank = cart.Get_rank()
    coords = cart.Get_coords(rank)
    nbr_up, nbr_down = cart.Shift(0, 1)       # axis 0: rows   (up = row_rank-1)
    nbr_left, nbr_right = cart.Shift(1, 1)    # axis 1: columns (left = col_rank-1)

    # Bind every rank to the single physical GPU before any device allocation.
    cuda.select_device(0)
    dev = cuda.get_current_device()
    gpu_name = dev.name.decode() if isinstance(dev.name, bytes) else str(dev.name)
    cc = tuple(dev.compute_capability)
    gpu_uuid = _query_gpu_uuid(dev)
    dev_attrs = {}
    for a in ("MULTIPROCESSOR_COUNT", "ASYNC_ENGINE_COUNT", "CONCURRENT_KERNELS",
              "MAX_SHARED_MEMORY_PER_BLOCK", "COMPUTE_MODE"):
        try:
            dev_attrs[a] = int(getattr(dev, a))
        except Exception:
            dev_attrs[a] = None

    # INSTANCE_SEED = first 64 bits of SHA-256(github_username || gpu_uuid)
    seed_material = f"{github_username}{gpu_uuid}"
    digest = hashlib.sha256(seed_material.encode("utf-8")).hexdigest()
    instance_seed = int(digest[:16], 16)      # 64-bit unsigned integer

    # All ranks must agree on the seed (same user, same GPU).
    s_min = cart.allreduce(instance_seed, op=MPI.MIN)
    s_max = cart.allreduce(instance_seed, op=MPI.MAX)
    assert s_min == s_max == instance_seed, "INSTANCE_SEED mismatch across ranks"

    ctx = Ctx()
    ctx.comm, ctx.cart, ctx.rank, ctx.size = comm, cart, rank, size
    ctx.coords = (int(coords[0]), int(coords[1]))
    ctx.nbr_up, ctx.nbr_down = nbr_up, nbr_down
    ctx.nbr_left, ctx.nbr_right = nbr_left, nbr_right
    ctx.github_username = github_username
    ctx.gpu_name, ctx.gpu_uuid, ctx.cc = gpu_name, gpu_uuid, cc
    ctx.dev_attrs = dev_attrs
    ctx.seed_digest = digest
    ctx.instance_seed = instance_seed
    ctx.hostname = MPI.Get_processor_name()
    return ctx


# ============================================================================
# TASK 2 - Seeded heterogeneous subdomain initialisation (rank-local only)
# ============================================================================
def initialize_subdomains(ctx):
    row_rank, col_rank = ctx.coords
    # Local index l in 0..257 -> global node index g = rank_offset + l in 0..513
    gi = row_rank * NY + np.arange(NY + 2, dtype=np.int64)   # rows  -> y
    gj = col_rank * NX + np.arange(NX + 2, dtype=np.int64)   # cols  -> x
    y = gi * DY
    x = gj * DX
    X, Y = np.meshgrid(x, y)          # both (258, 258); X varies along axis 1

    # Identical RNG stream on every rank -> globally consistent fields.
    rng = np.random.default_rng(ctx.instance_seed)

    # Diffusivity kappa(x,y) in [0.08, 0.24]
    ph_x, ph_y = rng.uniform(0.0, 2.0 * np.pi, size=2)
    kmid = 0.5 * (KAPPA_LO + KAPPA_HI)       # 0.16
    kamp = 0.5 * (KAPPA_HI - KAPPA_LO)       # 0.08
    kappa = kmid + kamp * np.sin(3.0 * np.pi * X + ph_x) * np.cos(2.0 * np.pi * Y + ph_y)

    # Initial Gaussian thermal pulses (centres / amplitudes / widths from seed)
    n_pulses = int(rng.integers(3, 5))       # 3 or 4 pulses
    pulses = []
    u0 = np.zeros(LOCAL_SHAPE, dtype=np.float64)
    for _ in range(n_pulses):
        cx, cy = rng.uniform(0.2, 0.8, size=2)
        amp = rng.uniform(0.6, 1.0)
        sig = rng.uniform(0.05, 0.09)
        u0 += amp * np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2.0 * sig * sig))
        pulses.append({"cx": float(cx), "cy": float(cy), "amp": float(amp), "sigma": float(sig)})

    # Localised (weak) source S(x,y)
    sx, sy = rng.uniform(0.3, 0.7, size=2)
    s_amp, s_sig = 0.05, 0.03
    src = s_amp * np.exp(-((X - sx) ** 2 + (Y - sy) ** 2) / (2.0 * s_sig * s_sig))

    # Homogeneous Dirichlet u = 0 on outer global edges (neighbour == PROC_NULL)
    if ctx.nbr_up == MPI.PROC_NULL:
        u0[0, :] = 0.0
    if ctx.nbr_down == MPI.PROC_NULL:
        u0[-1, :] = 0.0
    if ctx.nbr_left == MPI.PROC_NULL:
        u0[:, 0] = 0.0
    if ctx.nbr_right == MPI.PROC_NULL:
        u0[:, -1] = 0.0

    meta = {
        "kappa_phase": [float(ph_x), float(ph_y)],
        "pulses": pulses,
        "source": {"cx": float(sx), "cy": float(sy), "amp": s_amp, "sigma": s_sig},
        "local_x_range": [float(x[0]), float(x[-1])],
        "local_y_range": [float(y[0]), float(y[-1])],
    }
    return (np.ascontiguousarray(u0), np.ascontiguousarray(kappa),
            np.ascontiguousarray(src), meta)


# ============================================================================
# TASK 3 - GPU halo pack / unpack kernels and pinned asynchronous exchange
# ============================================================================
@cuda.jit
def pack_halos_kernel(u, send, ny, nx):
    """Collect the 4 owned border strips (strided columns!) into one contiguous vector."""
    k = cuda.grid(1)
    if k < nx:
        send[k] = u[1, k + 1]                       # top owned row    (row 1)
        send[nx + k] = u[ny, k + 1]                 # bottom owned row (row ny)
    if k < ny:
        send[2 * nx + k] = u[k + 1, 1]              # left owned col   (col 1)  - stride 258
        send[2 * nx + ny + k] = u[k + 1, nx]        # right owned col  (col nx) - stride 258


@cuda.jit
def unpack_halos_kernel(u, recv, ny, nx, has_up, has_down, has_left, has_right):
    """Scatter received contiguous vectors into ghost rows/cols 0 and n+1."""
    k = cuda.grid(1)
    if k < nx:
        if has_up != 0:
            u[0, k + 1] = recv[k]                   # ghost row 0
        if has_down != 0:
            u[ny + 1, k + 1] = recv[nx + k]         # ghost row ny+1
    if k < ny:
        if has_left != 0:
            u[k + 1, 0] = recv[2 * nx + k]          # ghost col 0
        if has_right != 0:
            u[k + 1, nx + 1] = recv[2 * nx + ny + k]  # ghost col nx+1


def _pack_grid():
    return (max(NX, NY) + PACK_THREADS - 1) // PACK_THREADS


def start_halo_exchange(ctx, u_d, st):
    """Stage A + B(MPI part): pack on GPU, async D2H to pinned, post Irecv/Isend."""
    b = ctx.bufs
    pack_halos_kernel[_pack_grid(), PACK_THREADS, st](u_d, b["d_send"], NY, NX)
    b["d_send"].copy_to_host(b["h_send"], stream=st)     # async DMA -> page-locked
    st.synchronize()                                       # host send buffers ready

    cart, hs, hr = ctx.cart, b["h_send"], b["h_recv"]
    reqs = [
        # receives first (pre-posted), tag-matched to the sender's direction
        cart.Irecv([hr[OFF_TOP:OFF_TOP + NX], MPI.DOUBLE], source=ctx.nbr_up, tag=TAG_SOUTHWARD),
        cart.Irecv([hr[OFF_BOT:OFF_BOT + NX], MPI.DOUBLE], source=ctx.nbr_down, tag=TAG_NORTHWARD),
        cart.Irecv([hr[OFF_LEFT:OFF_LEFT + NY], MPI.DOUBLE], source=ctx.nbr_left, tag=TAG_EASTWARD),
        cart.Irecv([hr[OFF_RIGHT:OFF_RIGHT + NY], MPI.DOUBLE], source=ctx.nbr_right, tag=TAG_WESTWARD),
        cart.Isend([hs[OFF_TOP:OFF_TOP + NX], MPI.DOUBLE], dest=ctx.nbr_up, tag=TAG_NORTHWARD),
        cart.Isend([hs[OFF_BOT:OFF_BOT + NX], MPI.DOUBLE], dest=ctx.nbr_down, tag=TAG_SOUTHWARD),
        cart.Isend([hs[OFF_LEFT:OFF_LEFT + NY], MPI.DOUBLE], dest=ctx.nbr_left, tag=TAG_WESTWARD),
        cart.Isend([hs[OFF_RIGHT:OFF_RIGHT + NY], MPI.DOUBLE], dest=ctx.nbr_right, tag=TAG_EASTWARD),
    ]
    return reqs


def finish_halo_exchange(ctx, u_d, reqs, st):
    """Stage C(MPI part): Waitall, async H2D from pinned, unpack into ghost cells."""
    b = ctx.bufs
    MPI.Request.Waitall(reqs)
    b["d_recv"].copy_to_device(b["h_recv"], stream=st)   # async DMA <- page-locked
    unpack_halos_kernel[_pack_grid(), PACK_THREADS, st](
        u_d, b["d_recv"], NY, NX,
        ctx.has_up, ctx.has_down, ctx.has_left, ctx.has_right)


def exchange_halos_async(ctx, u_d, st):
    """Complete pinned, stream-ordered halo exchange (used by the serialized path)."""
    reqs = start_halo_exchange(ctx, u_d, st)
    finish_halo_exchange(ctx, u_d, reqs, st)


# ============================================================================
# TASK 4 - Shared-memory tiled stencil kernel and dual-stream orchestrator
# ============================================================================
@cuda.jit
def tiled_stencil_kernel(u, u_new, kap, src, r_start, r_end, c_start, c_end,
                         dt, inv_dx2, inv_dy2, lam):
    """Update cells (i,j) with r_start<=i<=r_end, c_start<=j<=c_end.

    16x16 threads, each block stages an 18x18 tile of u (and of kappa) incl. a 1-cell apron.
    Only cells inside [r_start-1, r_end+1] x [c_start-1, c_end+1] are ever read, so the
    deep-interior launch never touches ghost cells that stream_boundary is filling.
    """
    su = cuda.shared.array(shape=(TILE_HALO, TILE_HALO), dtype=np.float64)
    sk = cuda.shared.array(shape=(TILE_HALO, TILE_HALO), dtype=np.float64)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    j = c_start + cuda.blockIdx.x * TILE + tx      # column (x direction)
    i = r_start + cuda.blockIdx.y * TILE + ty      # row    (y direction)
    sx = tx + 1
    sy = ty + 1
    r_lim = r_end + 1
    c_lim = c_end + 1

    if i <= r_lim and j <= c_lim:
        su[sy, sx] = u[i, j]                        # centre cell
        sk[sy, sx] = kap[i, j]
        if ty == 0:                                 # top apron row
            su[0, sx] = u[i - 1, j]
            sk[0, sx] = kap[i - 1, j]
        if ty == TILE - 1 and i + 1 <= r_lim:       # bottom apron row
            su[TILE + 1, sx] = u[i + 1, j]
            sk[TILE + 1, sx] = kap[i + 1, j]
        if tx == 0:                                 # left apron column
            su[sy, 0] = u[i, j - 1]
            sk[sy, 0] = kap[i, j - 1]
        if tx == TILE - 1 and j + 1 <= c_lim:       # right apron column
            su[sy, TILE + 1] = u[i, j + 1]
            sk[sy, TILE + 1] = kap[i, j + 1]

    cuda.syncthreads()                              # tile complete before any stencil math

    if i <= r_end and j <= c_end:
        uc = su[sy, sx]
        kc = sk[sy, sx]
        k_e = 0.5 * (sk[sy, sx + 1] + kc)           # kappa_{i,j+1/2}
        k_w = 0.5 * (sk[sy, sx - 1] + kc)           # kappa_{i,j-1/2}
        k_s = 0.5 * (sk[sy + 1, sx] + kc)           # kappa_{i+1/2,j}
        k_n = 0.5 * (sk[sy - 1, sx] + kc)           # kappa_{i-1/2,j}
        flux_x = (k_e * (su[sy, sx + 1] - uc) - k_w * (uc - su[sy, sx - 1])) * inv_dx2
        flux_y = (k_s * (su[sy + 1, sx] - uc) - k_n * (uc - su[sy - 1, sx])) * inv_dy2
        u_new[i, j] = uc + dt * (flux_x + flux_y - lam * uc * uc * uc + src[i, j])


BOX_FULL = (1, NY, 1, NX)
BOX_DEEP = (2, NY - 1, 2, NX - 1)                   # [2..255, 2..255]
BOX_RING = [
    (1, 1, 1, NX),                                  # row 1
    (NY, NY, 1, NX),                                # row ny
    (2, NY - 1, 1, 1),                              # col 1   (corners already covered)
    (2, NY - 1, NX, NX),                            # col nx
]


def launch_stencil(ctx, st, u_d, un_d, box):
    r0, r1, c0, c1 = box
    rows, cols = r1 - r0 + 1, c1 - c0 + 1
    grid = ((cols + TILE - 1) // TILE, (rows + TILE - 1) // TILE)
    tiled_stencil_kernel[grid, (TILE, TILE), st](
        u_d, un_d, ctx.kappa_d, ctx.src_d, r0, r1, c0, c1,
        ctx.dt, ctx.inv_dx2, ctx.inv_dy2, LAMBDA)


def step_serialized(ctx):
    """Reference path: exchange -> full stencil, everything on one stream, no overlap."""
    u_d, un_d = ctx.fields[ctx.cur], ctx.fields[1 - ctx.cur]
    sb = ctx.stream_boundary
    exchange_halos_async(ctx, u_d, sb)
    launch_stencil(ctx, sb, u_d, un_d, BOX_FULL)
    sb.synchronize()
    ctx.cur = 1 - ctx.cur


def step_overlapped(ctx):
    """Dual-stream pipeline (Stages A/B/C)."""
    u_d, un_d = ctx.fields[ctx.cur], ctx.fields[1 - ctx.cur]
    sb, si = ctx.stream_boundary, ctx.stream_interior
    # Stage A + B: pack, D2H, sync, post Irecv/Isend
    reqs = start_halo_exchange(ctx, u_d, sb)
    # Stage B: deep interior compute runs concurrently with the MPI exchange
    launch_stencil(ctx, si, u_d, un_d, BOX_DEEP)
    # Stage C: Waitall, H2D, unpack, then the 4 perimeter rings on stream_boundary
    finish_halo_exchange(ctx, u_d, reqs, sb)
    for box in BOX_RING:
        launch_stencil(ctx, sb, u_d, un_d, box)
    sb.synchronize()
    si.synchronize()
    ctx.cur = 1 - ctx.cur


# ============================================================================
# TASK 5 - Reductions, PTX audit, benchmark, manifest
# ============================================================================
def global_sum(ctx, value):
    out = np.zeros(1, dtype=np.float64)
    ctx.cart.Allreduce(np.array([value], dtype=np.float64), out, op=MPI.SUM)
    return float(out[0])


def global_max(ctx, value):
    out = np.zeros(1, dtype=np.float64)
    ctx.cart.Allreduce(np.array([value], dtype=np.float64), out, op=MPI.MAX)
    return float(out[0])


def global_min(ctx, value):
    out = np.zeros(1, dtype=np.float64)
    ctx.cart.Allreduce(np.array([value], dtype=np.float64), out, op=MPI.MIN)
    return float(out[0])


def diagnostics(ctx):
    """Global energy E = sum u dx dy and L2 norm of the last increment ||u^n - u^{n-1}||_2."""
    cur = ctx.fields[ctx.cur].copy_to_host()
    prev = ctx.fields[1 - ctx.cur].copy_to_host()   # holds u^{n-1} after the swap
    own = cur[1:NY + 1, 1:NX + 1]
    own_prev = prev[1:NY + 1, 1:NX + 1]
    e_loc = float(np.sum(own) * DX * DY)
    r_loc = float(np.sum((own - own_prev) ** 2))
    energy = global_sum(ctx, e_loc)
    resid = math.sqrt(global_sum(ctx, r_loc))
    umax = global_max(ctx, float(np.max(own)))
    return energy, resid, umax


def reset_fields(ctx):
    ctx.fields[0].copy_to_device(ctx.u0_host)
    ctx.fields[1].copy_to_device(ctx.u0_host)
    ctx.bufs["h_recv"][:] = 0.0
    ctx.cur = 0
    cuda.synchronize()


def run_mode(ctx, mode):
    step = step_serialized if mode == "serialized" else step_overlapped
    reset_fields(ctx)
    trace = []
    e0, _, umax0 = diagnostics(ctx)
    trace.append({"step": 0, "energy": e0, "l2_residual": 0.0, "u_max": umax0})
    root_print(ctx, f"  [{mode:>10}] step {0:4d} | E = {e0:.15e} | L2 = {0.0:.6e} | max u = {umax0:.6f}")

    ev_ms_total, wall_s_total = 0.0, 0.0
    n_chunks = N_STEPS // DIAG_EVERY
    for c in range(n_chunks):
        cuda.synchronize()
        ctx.cart.Barrier()
        ev_start = cuda.event(timing=True)
        ev_stop = cuda.event(timing=True)
        ev_start.record()
        t0 = time.perf_counter()
        for _ in range(DIAG_EVERY):
            step(ctx)
        ev_stop.record()
        ev_stop.synchronize()
        t1 = time.perf_counter()
        ev_ms_total += cuda.event_elapsed_time(ev_start, ev_stop)
        wall_s_total += (t1 - t0)

        s = (c + 1) * DIAG_EVERY
        e, r, um = diagnostics(ctx)
        trace.append({"step": s, "energy": e, "l2_residual": r, "u_max": um})
        root_print(ctx, f"  [{mode:>10}] step {s:4d} | E = {e:.15e} | L2 = {r:.6e} | max u = {um:.6f}")

    final = ctx.fields[ctx.cur].copy_to_host()
    ev_ms = global_max(ctx, ev_ms_total)            # slowest rank defines time-to-solution
    wall_ms = global_max(ctx, wall_s_total * 1e3)
    return final, trace, ev_ms, wall_ms


def ptx_audit(kernel):
    asm = kernel.inspect_asm()
    if isinstance(asm, dict):
        ptx = "\n".join(str(v) for v in asm.values())
    else:
        ptx = str(asm)
    counts = {
        "st.shared.f64": ptx.count("st.shared.f64"),
        "ld.shared.f64": ptx.count("ld.shared.f64"),
        "ld.shared.v2.f64": ptx.count("ld.shared.v2.f64"),
        "st.shared (any)": len(re.findall(r"\bst\.shared\.", ptx)),
        "ld.shared (any)": len(re.findall(r"\bld\.shared\.", ptx)),
        "bar.sync": ptx.count("bar.sync"),
        "barrier.sync": ptx.count("barrier.sync"),
        "ld.global.f64 (any)": len(re.findall(r"\bld\.global(\.nc)?\.f64", ptx)),
        "st.global.f64": ptx.count("st.global.f64"),
        "fma.rn.f64": ptx.count("fma.rn.f64"),
        "ptx_lines": ptx.count("\n") + 1,
    }
    passed = (counts["st.shared (any)"] > 0 and counts["ld.shared (any)"] > 0
              and (counts["bar.sync"] + counts["barrier.sync"]) > 0)
    return counts, passed, ptx


def _file_sha256(path):
    try:
        with open(path, "rb") as f:
            return hashlib.sha256(f.read()).hexdigest()
    except Exception:
        return None


def run_solver():
    github_username = os.environ.get("GITHUB_USERNAME", "")
    ctx = init_distributed_context(github_username)

    root_print(ctx, "=" * 78)
    root_print(ctx, " Distributed MPI + Numba-CUDA anisotropic reaction-diffusion solver")
    root_print(ctx, f" Student: {STUDENT_NAME} | ID: {STUDENT_ID} | GitHub: {ctx.github_username}")
    root_print(ctx, "=" * 78)
    root_print(ctx, f"[Task 1] MPI ranks = {ctx.size} | Cartesian dims = {PROC_DIMS} | periods = {PERIODS}")
    root_print(ctx, f"[Task 1] MPI library: {MPI.Get_library_version().splitlines()[0].strip()}")
    root_print(ctx, f"[Task 1] mpi4py {sys.modules['mpi4py'].__version__} | numba {numba.__version__} | numpy {np.__version__}")
    root_print(ctx, f"[Task 1] GPU: {ctx.gpu_name} | CC {ctx.cc[0]}.{ctx.cc[1]} | UUID {ctx.gpu_uuid}")
    root_print(ctx, f"[Task 1] GPU attributes: {ctx.dev_attrs}")
    root_print(ctx, f"[Task 1] SHA-256('{ctx.github_username}' + GPU UUID) = {ctx.seed_digest}")
    root_print(ctx, f"[Task 1] INSTANCE_SEED (64-bit) = {ctx.instance_seed}  (0x{ctx.instance_seed:016x})")

    def nb(v):
        return "PROC_NULL" if v == MPI.PROC_NULL else str(v)
    ordered_print(ctx, (f"[Task 1] rank {ctx.rank} coords={ctx.coords} host={ctx.hostname} "
                        f"up={nb(ctx.nbr_up)} down={nb(ctx.nbr_down)} "
                        f"left={nb(ctx.nbr_left)} right={nb(ctx.nbr_right)} | cuda device 0"))

    ctx.has_up = int(ctx.nbr_up != MPI.PROC_NULL)
    ctx.has_down = int(ctx.nbr_down != MPI.PROC_NULL)
    ctx.has_left = int(ctx.nbr_left != MPI.PROC_NULL)
    ctx.has_right = int(ctx.nbr_right != MPI.PROC_NULL)

    # ---------------- Task 2 ----------------
    u0, kappa, src, init_meta = initialize_subdomains(ctx)
    assert u0.shape == LOCAL_SHAPE and kappa.shape == LOCAL_SHAPE and src.shape == LOCAL_SHAPE
    ctx.u0_host = u0
    k_max = global_max(ctx, float(kappa.max()))
    k_min = global_min(ctx, float(kappa.min()))
    dt_max = DX * DX / (4.0 * k_max)
    ctx.dt = CFL_SAFETY * dt_max
    ctx.inv_dx2 = 1.0 / (DX * DX)
    ctx.inv_dy2 = 1.0 / (DY * DY)
    root_print(ctx, f"[Task 2] local padded array shape = {LOCAL_SHAPE} (owned {NY}x{NX}) | global {NY_GLOBAL}x{NX_GLOBAL}")
    root_print(ctx, f"[Task 2] kappa range (global Allreduce) = [{k_min:.6f}, {k_max:.6f}]")
    root_print(ctx, f"[Task 2] dx = dy = 1/513 = {DX:.10e} | dt_max = dx^2/(4 kappa_max) = {dt_max:.10e}")
    root_print(ctx, f"[Task 2] dt = {CFL_SAFETY} * dt_max = {ctx.dt:.10e} | dt/dt_max = {ctx.dt / dt_max:.3f}")
    root_print(ctx, f"[Task 2] {len(init_meta['pulses'])} seeded Gaussian pulses:")
    for p in init_meta["pulses"]:
        root_print(ctx, f"           centre=({p['cx']:.4f},{p['cy']:.4f}) amp={p['amp']:.4f} sigma={p['sigma']:.4f}")
    s = init_meta["source"]
    root_print(ctx, f"[Task 2] source S: centre=({s['cx']:.4f},{s['cy']:.4f}) amp={s['amp']} sigma={s['sigma']}")
    ordered_print(ctx, (f"[Task 2] rank {ctx.rank} x in [{init_meta['local_x_range'][0]:.5f},"
                        f"{init_meta['local_x_range'][1]:.5f}] y in [{init_meta['local_y_range'][0]:.5f},"
                        f"{init_meta['local_y_range'][1]:.5f}] | local sum(u0) = {u0[1:-1, 1:-1].sum():.6f}"))

    # ---------------- device allocation (rank-local only) ----------------
    ctx.fields = [cuda.to_device(u0), cuda.to_device(u0)]
    ctx.kappa_d = cuda.to_device(kappa)
    ctx.src_d = cuda.to_device(src)
    ctx.stream_boundary = cuda.stream()
    ctx.stream_interior = cuda.stream()
    ctx.bufs = {
        "d_send": cuda.device_array(HALO_LEN, dtype=np.float64),
        "d_recv": cuda.device_array(HALO_LEN, dtype=np.float64),
        "h_send": cuda.pinned_array(HALO_LEN, dtype=np.float64),
        "h_recv": cuda.pinned_array(HALO_LEN, dtype=np.float64),
    }
    ctx.bufs["h_send"][:] = 0.0
    ctx.bufs["h_recv"][:] = 0.0
    ctx.cur = 0
    root_print(ctx, f"[Task 3] pinned host halo buffers: 2 x {HALO_LEN} float64 = {2 * HALO_LEN * 8} bytes per rank")
    root_print(ctx, f"[Task 3] halo payload per step per rank: {HALO_LEN * 8} B D2H + {HALO_LEN * 8} B H2D")

    # ---------------- JIT warm-up (excluded from timing) ----------------
    t_jit = time.perf_counter()
    for _ in range(2):
        step_serialized(ctx)
    for _ in range(2):
        step_overlapped(ctx)
    cuda.synchronize()
    ctx.cart.Barrier()
    root_print(ctx, f"[Task 4] JIT warm-up done in {time.perf_counter() - t_jit:.2f} s "
                    f"(tile {TILE}x{TILE} threads, shared tile {TILE + 2}x{TILE + 2} float64)")

    # ---------------- Task 5: benchmark both modes ----------------
    root_print(ctx, f"[Task 5] Running {N_STEPS} steps - SERIALIZED (single stream, no overlap)")
    fin_ser, trace_ser, ser_ms, ser_wall = run_mode(ctx, "serialized")
    root_print(ctx, f"[Task 5] Running {N_STEPS} steps - OVERLAPPED (stream_boundary + stream_interior)")
    fin_ovl, trace_ovl, ovl_ms, ovl_wall = run_mode(ctx, "overlapped")

    local_diff = float(np.max(np.abs(fin_ser[1:-1, 1:-1] - fin_ovl[1:-1, 1:-1])))
    parity_max_diff = global_max(ctx, local_diff)
    parity_ok = parity_max_diff < PARITY_TOL
    energies = [t["energy"] for t in trace_ovl]
    monotonic = all(energies[k + 1] <= energies[k] for k in range(len(energies) - 1))
    speedup = ser_ms / ovl_ms if ovl_ms > 0 else float("nan")

    root_print(ctx, "-" * 78)
    root_print(ctx, f"[Task 5] serialized_ms (CUDA events, max over ranks) = {ser_ms:.3f} ms "
                    f"({ser_ms / N_STEPS:.4f} ms/step) | wall = {ser_wall:.3f} ms")
    root_print(ctx, f"[Task 5] overlapped_ms (CUDA events, max over ranks) = {ovl_ms:.3f} ms "
                    f"({ovl_ms / N_STEPS:.4f} ms/step) | wall = {ovl_wall:.3f} ms")
    root_print(ctx, f"[Task 5] overlap speedup serialized/overlapped = {speedup:.3f}x")
    root_print(ctx, f"[Task 5] parity max |u_ser - u_ovl| = {parity_max_diff:.3e} "
                    f"(< {PARITY_TOL:.0e}: {parity_ok})")
    root_print(ctx, f"[Task 5] energy monotonically non-increasing over steps 0..{N_STEPS}: {monotonic}")
    for k in range(1, len(energies)):
        root_print(ctx, f"           dE[{trace_ovl[k - 1]['step']:3d}->{trace_ovl[k]['step']:3d}] = "
                        f"{energies[k] - energies[k - 1]:+.6e}")

    # ---------------- PTX audit ----------------
    counts, ptx_ok, ptx_text = ptx_audit(tiled_stencil_kernel)
    root_print(ctx, "[Task 5] PTX audit of tiled_stencil_kernel.inspect_asm():")
    for k, v in counts.items():
        root_print(ctx, f"           {k:<22s} : {v}")
    root_print(ctx, f"[Task 5] ptx_audit_passed = {ptx_ok}")

    # ---------------- manifest (rank 0) ----------------
    if ctx.rank == 0:
        with open("tiled_stencil_kernel.ptx", "w") as f:
            f.write(ptx_text)
        topo = []
        for r in range(ctx.size):
            rc = ctx.cart.Get_coords(r)
            topo.append({"rank": r, "coords": [int(rc[0]), int(rc[1])]})
        manifest = {
            "student_name": STUDENT_NAME,
            "student_id": STUDENT_ID,
            "GITHUB_USERNAME": ctx.github_username,
            "github_username": ctx.github_username,
            "repository": f"mpi-cuda-stencil-{ctx.github_username}",
            "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "gpu_name": ctx.gpu_name,
            "gpu_uuid": ctx.gpu_uuid,
            "compute_capability": f"{ctx.cc[0]}.{ctx.cc[1]}",
            "gpu_attributes": ctx.dev_attrs,
            "seed_sha256": ctx.seed_digest,
            "INSTANCE_SEED": str(ctx.instance_seed),
            "instance_seed_hex": f"0x{ctx.instance_seed:016x}",
            "mpi_ranks": ctx.size,
            "cart_dims": PROC_DIMS,
            "periods": PERIODS,
            "rank_topology": topo,
            "mpi_library": MPI.Get_library_version().splitlines()[0].strip(),
            "versions": {"python": platform.python_version(), "numpy": np.__version__,
                         "numba": numba.__version__, "mpi4py": sys.modules["mpi4py"].__version__},
            "global_grid": [NY_GLOBAL, NX_GLOBAL],
            "local_padded_shape": list(LOCAL_SHAPE),
            "threadblock": [TILE, TILE],
            "shared_tile": [TILE + 2, TILE + 2],
            "dx": DX, "dy": DY,
            "kappa_min": k_min, "kappa_max": k_max,
            "dt": ctx.dt, "dt_max_cfl": dt_max, "cfl_fraction": ctx.dt / dt_max,
            "lambda": LAMBDA,
            "n_steps": N_STEPS,
            "initial_conditions": init_meta["pulses"],
            "source": init_meta["source"],
            "energy_trace_serialized": trace_ser,
            "energy_trace_overlapped": trace_ovl,
            "energy_monotonic_nonincreasing": monotonic,
            "serialized_ms": ser_ms,
            "overlapped_ms": ovl_ms,
            "serialized_wall_ms": ser_wall,
            "overlapped_wall_ms": ovl_wall,
            "overlap_speedup": speedup,
            "parity_max_diff": parity_max_diff,
            "parity_passed": parity_ok,
            "ptx_counts": counts,
            "ptx_audit_passed": ptx_ok,
            "halo_bytes_per_step_per_rank": {"d2h": HALO_LEN * 8, "h2d": HALO_LEN * 8},
            "forbidden_collectives_on_fields": False,
            "solver_sha256": _file_sha256(os.path.abspath(__file__)),
            "tasks_completed": {
                "task1_topology_seed": True,
                "task2_initialization": True,
                "task3_halo_exchange": True,
                "task4_tiled_stencil_overlap": True,
                "task5_reductions_ptx_benchmark": bool(parity_ok and ptx_ok),
            },
        }
        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        print("-" * 78, flush=True)
        print("[Task 5] submission_manifest.json written", flush=True)
        if parity_ok and ptx_ok:
            print("ALL CHECKS PASSED", flush=True)
        else:
            print("SOME CHECKS FAILED - see above", flush=True)

    ctx.cart.Barrier()
    return 0


if __name__ == "__main__":
    sys.exit(run_solver())

Writing distributed_cuda_solver.py


## Cell 4 — Launch 4 MPI ranks (Tasks 1–5)

In [4]:
import os, sys, subprocess

env = dict(os.environ,
           GITHUB_USERNAME=GITHUB_USERNAME, STUDENT_NAME=STUDENT_NAME, STUDENT_ID=STUDENT_ID,
           OMPI_ALLOW_RUN_AS_ROOT="1", OMPI_ALLOW_RUN_AS_ROOT_CONFIRM="1",
           PYTHONUNBUFFERED="1", NUMBA_CUDA_LOW_OCCUPANCY_WARNINGS="0")
cmd = ["mpirun", "--allow-run-as-root", "--oversubscribe",
       "--mca", "btl_vader_single_copy_mechanism", "none",
       "-np", "4", sys.executable, "distributed_cuda_solver.py"]
print("$", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
with open("run_log.txt", "w") as log:
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
rc = proc.wait()
print(f"\nmpirun exit code: {rc}")
assert rc == 0, "Solver failed - see the log above"

$ mpirun --allow-run-as-root --oversubscribe --mca btl_vader_single_copy_mechanism none -np 4 /usr/bin/python3 distributed_cuda_solver.py
 Distributed MPI + Numba-CUDA anisotropic reaction-diffusion solver
 Student: Ainat Yermurat | ID: 230103161 | GitHub: yermy-bc
[Task 1] MPI ranks = 4 | Cartesian dims = [2, 2] | periods = [False, False]
[Task 1] MPI library: Open MPI v4.1.6, package: Debian OpenMPI, ident: 4.1.6, repo rev: v4.1.6, Sep 30, 2023 
[Task 1] mpi4py 4.1.2 | numba 0.64.0 | numpy 2.4.6
[Task 1] GPU: Tesla T4 | CC 7.5 | UUID GPU-abeebd7d-e34b-f585-83bc-9af29ec895f6
[Task 1] GPU attributes: {'MULTIPROCESSOR_COUNT': 40, 'ASYNC_ENGINE_COUNT': 3, 'CONCURRENT_KERNELS': 1, 'MAX_SHARED_MEMORY_PER_BLOCK': 49152, 'COMPUTE_MODE': 0}
[Task 1] SHA-256('yermy-bc' + GPU UUID) = e200e54fef78c6d6305a9624a42a16f343768a75de08779543911df23b0efe28
[Task 1] INSTANCE_SEED (64-bit) = 16285268384054560470  (0xe200e54fef78c6d6)
[Task 1] rank 0 coords=(0, 0) host=1a1482720690 up=PROC_NULL down=2 left

## Cell 5 — Manifest verification

In [5]:
import json
with open("submission_manifest.json") as f:
    m = json.load(f)

print(f"GITHUB_USERNAME   : {m['GITHUB_USERNAME']}")
print(f"student           : {m['student_name']} ({m['student_id']})")
print(f"GPU               : {m['gpu_name']} | CC {m['compute_capability']} | {m['gpu_uuid']}")
print(f"INSTANCE_SEED     : {m['INSTANCE_SEED']} ({m['instance_seed_hex']})")
print(f"mpi_ranks         : {m['mpi_ranks']}  cart_dims={m['cart_dims']}  periods={m['periods']}")
print(f"kappa range       : [{m['kappa_min']:.6f}, {m['kappa_max']:.6f}]")
print(f"dt / dt_max       : {m['dt']:.6e} / {m['dt_max_cfl']:.6e}  ({m['cfl_fraction']:.2f})")
print(f"serialized_ms     : {m['serialized_ms']:.3f}")
print(f"overlapped_ms     : {m['overlapped_ms']:.3f}")
print(f"overlap speedup   : {m['overlap_speedup']:.3f}x")
print(f"parity_max_diff   : {m['parity_max_diff']:.3e}")
print(f"ptx_counts        : {m['ptx_counts']}")
print(f"ptx_audit_passed  : {m['ptx_audit_passed']}")
print("energy trace (overlapped):")
for t in m["energy_trace_overlapped"]:
    print(f"   step {t['step']:3d}  E = {t['energy']:.15e}  L2 = {t['l2_residual']:.6e}")

assert m["GITHUB_USERNAME"] == GITHUB_USERNAME
assert m["mpi_ranks"] == 4 and m["cart_dims"] == [2, 2]
assert m["ptx_audit_passed"] is True
assert m["parity_max_diff"] < 1e-12
print("\nAll pre-submission checks satisfied.")

GITHUB_USERNAME   : yermy-bc
student           : Ainat Yermurat (230103161)
GPU               : Tesla T4 | CC 7.5 | GPU-abeebd7d-e34b-f585-83bc-9af29ec895f6
INSTANCE_SEED     : 16285268384054560470 (0xe200e54fef78c6d6)
mpi_ranks         : 4  cart_dims=[2, 2]  periods=[False, False]
kappa range       : [0.080002, 0.239998]
dt / dt_max       : 3.166559e-06 / 3.958199e-06  (0.80)
serialized_ms     : 963.371
overlapped_ms     : 1393.861
overlap speedup   : 0.691x
parity_max_diff   : 0.000e+00
ptx_counts        : {'st.shared.f64': 10, 'ld.shared.f64': 10, 'ld.shared.v2.f64': 0, 'st.shared (any)': 10, 'ld.shared (any)': 10, 'bar.sync': 1, 'barrier.sync': 0, 'ld.global.f64 (any)': 11, 'st.global.f64': 1, 'fma.rn.f64': 2, 'ptx_lines': 369}
ptx_audit_passed  : True
energy trace (overlapped):
   step   0  E = 6.220141891062554e-02  L2 = 0.000000e+00
   step 100  E = 6.212800344479495e-02  L2 = 1.583920e-02
   step 200  E = 6.208424746570877e-02  L2 = 1.490631e-02
   step 300  E = 6.2045560926803

## Cell 6 — PTX excerpt (shared-memory traffic and barrier)

In [6]:
with open("tiled_stencil_kernel.ptx") as f:
    lines = f.read().splitlines()
hits = [l.strip() for l in lines if ("shared" in l and ("ld." in l or "st." in l)) or "bar.sync" in l or "barrier.sync" in l]
print(f"{len(hits)} shared/barrier instructions in tiled_stencil_kernel PTX:")
print("\n".join(hits))

21 shared/barrier instructions in tiled_stencil_kernel PTX:
st.shared.f64 	[%rd55], %fd5;
st.shared.f64 	[%rd63], %fd6;
st.shared.f64 	[%rd14], %fd7;
st.shared.f64 	[%rd15], %fd8;
st.shared.f64 	[%rd14+2448], %fd9;
st.shared.f64 	[%rd15+2448], %fd10;
st.shared.f64 	[%rd16], %fd11;
st.shared.f64 	[%rd17], %fd12;
st.shared.f64 	[%rd16+136], %fd13;
st.shared.f64 	[%rd17+136], %fd14;
bar.sync 	0;
ld.shared.f64 	%fd15, [%rd130];
ld.shared.f64 	%fd16, [%rd124];
ld.shared.f64 	%fd19, [%rd135];
ld.shared.f64 	%fd22, [%rd142];
ld.shared.f64 	%fd25, [%rd148];
ld.shared.f64 	%fd28, [%rd149];
ld.shared.f64 	%fd29, [%rd122];
ld.shared.f64 	%fd32, [%rd150];
ld.shared.f64 	%fd36, [%rd151];
ld.shared.f64 	%fd39, [%rd152];


## Q1 — Memory coalescing & strided column packing

The padded field is a C-contiguous `float64` array of shape `258 × 258`, so element `u[i, j]` lives at byte offset
`(258·i + j)·8`. A row strip `u[1, 1:257]` is 256 consecutive doubles (2 048 B contiguous), but the column strip
`u[1:257, 1]` has a **stride of 258 × 8 = 2 064 B** between consecutive elements.

* **Warp level.** When a warp of 32 threads reads a row, the 32 addresses fall into 256 contiguous bytes = 8 32-byte sectors (100 % sector efficiency). When the same warp reads a column, every thread touches a *different* 32-byte sector 2 064 B apart: 32 sectors / transactions for 256 useful bytes — only 8 of every 32 fetched bytes are used (25 % efficiency), and the accesses span 32 different DRAM rows.
* **DMA level.** `cudaMemcpyAsync` (what `copy_to_host(..., stream=)` issues) moves **one contiguous byte range**. A column is 256 disjoint 8-byte pieces, so it cannot be one DMA descriptor; it would need 256 tiny copies (each paying ~µs of launch/PCIe latency) or a pitched 2-D copy with an 8-byte width. In addition, for the H2D direction a contiguous range covering a ghost column would also overwrite owned interior cells, i.e. it would be *wrong*, not only slow.

**Extra bytes for unstrided bounding-box copies** (per rank, per step, D2H):

| strategy | bytes moved | useful | wasted |
|---|---|---|---|
| packed vector (`pack_halos_kernel`) — 4 strips × 256 × 8 B | **8 192 B** | 8 192 B | 0 |
| contiguous span `u[1,1] … u[256,1]` = 255·258+1 = 65 791 doubles | 526 328 B per column | 2 048 B | **524 280 B per column** |
| both columns (left + right) | 1 052 656 B | 4 096 B | **1 048 560 B** |

So per step each rank would push ≈ **1.05 MB of extra PCIe traffic** (≈ 128× the packed payload of 8 192 B),
and over my 400-step run on 4 ranks that is 1 048 560 × 400 × 4 ≈ **1.68 GB** of avoidable D2H traffic
(doubled again if the same trick were attempted for H2D). My packed exchange moved only
8 192 B D2H + 8 192 B H2D per rank per step (printed in my run as `[Task 3] halo payload per step per rank: 8192 B D2H + 8192 B H2D`).
The pack kernel turns the strided gather into a GPU-side operation (L2/HBM bandwidth ≈ 300 GB/s on T4) and lets one
contiguous 8 KB DMA cross PCIe.

## Q2 — Shared-memory tile indexing & bank conflicts

Tile: `su[18][18]` (and the identical `sk` tile for κ), `float64`, `sx = tx+1`, `sy = ty+1`. For a block in the
interior of the launch box:

| thread | writes (`su` and `sk`) | reads in the stencil |
|---|---|---|
| `(tx=0, ty=0)` | `[1,1]` centre, `[0,1]` top apron (`ty==0`), `[1,0]` left apron (`tx==0`) | `[1,1]`, `[1,2]`, `[1,0]`, `[2,1]`, `[0,1]` |
| `(tx=15, ty=15)` | `[16,16]` centre, `[17,16]` bottom apron (`ty==15`), `[16,17]` right apron (`tx==15`) | `[16,16]`, `[16,17]`, `[16,15]`, `[17,16]`, `[15,16]` |

(The 4 corner cells `[0,0],[0,17],[17,0],[17,17]` are never written or read — the 5-point stencil does not need them.)

**Bank mapping on CC 7.5** (32 banks × 4 B): a double at linear index `k = 18·sy + sx` occupies 32-bit words
`2k, 2k+1`, i.e. banks `(2k) mod 32` and `(2k+1) mod 32`. Examples: `su[1,1]` → k = 19 → banks 6/7;
`su[16,16]` → k = 304 → banks 0/1. One tile row is 18 × 8 = 144 B = 36 words, so moving one row down shifts the bank by **4**.

* **Along a row (what the kernel does).** Thread linear id = `16·ty + tx`, so warp *w* holds tile rows `ty = 2w` and `2w+1`, 16 threads each. A 64-bit warp request is serviced as two half-warp phases of 16 × 8 B = 128 B. Each phase is exactly one tile row: 16 consecutive doubles → 32 consecutive words → all 32 banks exactly once → **no bank conflict**. This holds for every stencil read (`[sy,sx±1]`, `[sy±1,sx]`), because inside a half-warp `sx` varies and `sy` is constant.
* **Along a column (hypothetical).** If 16 threads of a half-warp read 16 different rows of the same column, word stride = 36 → bank stride 4 → only 32/4 = 8 distinct bank pairs for 16 threads → **2-way bank conflict** (threads `sy` and `sy+8` hit the same banks). The 18-wide padding does not fix it for doubles (it would need a stride that is odd in 8-byte units, e.g. 17).
* **Apron columns in my kernel.** Only `tx==0` / `tx==15` threads write `su[sy,0]` / `su[sy,17]`; per warp these are 2 threads (`ty=2w`, `2w+1`) that sit in different half-warps → no conflict, only branch divergence.

Conclusion: the row-major 16 × 16 → 18 × 18 mapping is conflict-free for every instruction my kernel actually issues; 64-bit **column** sweeps would be 2-way conflicted.

This is confirmed by my run's PTX (Cell 6): the apron stores use offsets `+2448` B (= 17 rows × 144 B) and `+136` B (= 17 × 8 B), exactly the 144-byte row pitch of the 18 × 18 `float64` tile used in the bank calculation above.


## Q3 — PTX audit & warp synchronization

Counts reported by `tiled_stencil_kernel.inspect_asm()` in my run (`[Task 5] PTX audit` block, `ptx_counts` in `submission_manifest.json`, 369 PTX lines in total):

| instruction | count | what it is |
|---|---|---|
| `st.shared.f64` | **10** | 2 tiles (`su` for u, `sk` for κ) × (1 centre + 4 apron writes: top, bottom, left, right) |
| `ld.shared.f64` | **10** | 2 centre reads (`uc`, `kc`) + 4 neighbour reads of `su` + 4 neighbour reads of `sk` |
| `bar.sync` | **1** | the single `cuda.syncthreads()` → `bar.sync 0` |
| `ld.shared.v2.f64` / `barrier.sync` | 0 / 0 | no vectorised shared loads, no alternative barrier form |
| `ld.global.f64` | 11 | 10 tile-fill loads of `u`/`κ` + 1 load of `S[i,j]` |
| `st.global.f64` | 1 | the single write to `u_new[i,j]` |

`ptx_audit_passed = true`.

The PTX listing in Cell 6 shows the exact ordering the algorithm requires: **all 10 `st.shared.f64` → `bar.sync 0` → all 10 `ld.shared.f64`**. The apron offsets are visible in the addressing: `[%rd14+2448]` is the bottom apron row (2448 B = 17 rows × 144 B, i.e. `su[17, sx]` relative to `su[0, sx]`), and `[%rd16+136]` is the right apron column (136 B = 17 × 8 B, i.e. `su[sy, 17]` relative to `su[sy, 0]`). Every stencil operand after the barrier comes from shared memory; global memory is read only once per tile cell. Without the tile, each cell would need 5 loads of u + 5 of κ from global memory.

**Hazard if `cuda.syncthreads()` is omitted.** A 16 × 16 block is 8 warps; warp *w* holds tile rows `ty = 2w, 2w+1`. Thread `(tx, ty=2w+1)` reads `su[sy+1, sx]` = tile row `2w+3`, and that row is **written by warp w+1**. The apron rows 0/17 are written only by warps 0 and 7, and the apron columns 0/17 only by lanes `tx = 0/15`. Warps are scheduled independently, so without the barrier warp *w* can reach its `ld.shared` before warp *w+1* has executed its `st.shared`. This is a **read-after-write (RAW) race across warps**. The load then returns whatever was in that shared-memory location before: shared memory is not zero-initialised, and it holds left-over data from the previous block resident on that SM. The update is computed from garbage, and the result depends on timing. Serialized and overlapped runs would then diverge, and my parity check (`parity_max_diff = 0.000e+00`) would fail. Since Volta's independent thread scheduling, even the intra-warp case (lane `tx=0` writes `su[sy,0]`, lane `tx=1` reads it) is not guaranteed safe without the barrier. With the single `bar.sync 0`, every store of the load phase is visible to all 256 threads before any of them starts the arithmetic.

## Q4 — Multi-process context sharing on a single physical GPU

**Measured (CUDA events, 400 steps, max over ranks):**

| mode | total | per step |
|---|---|---|
| `serialized_ms` | **963.371 ms** | 2.408 ms |
| `overlapped_ms` | **1393.861 ms** | 3.485 ms |
| speed-up serialized / overlapped | **0.691×** (overlap is ≈ 45 % slower) | |

The two modes produced bit-identical results (`parity_max_diff = 0.000e+00`), so the difference is purely scheduling overhead.

**How 4 processes share one T4.** `mpirun -np 4` creates 4 OS processes. Each calls `cuda.select_device(0)` and gets its **own CUDA context** on the same Tesla T4 (40 SMs, `ASYNC_ENGINE_COUNT = 3`, `COMPUTE_MODE = 0` = Default/shared, no MPS daemon on Colab). Without MPS the driver **time-slices** the GPU between contexts. Kernels from different contexts never execute concurrently; only one context occupies the SMs at a time, and each switch costs a context save/restore. The copy engines are also shared and arbitrated through the same scheduler. Concurrency between `stream_boundary` and `stream_interior` is therefore possible only *within* one rank's time slice. On top of that, all 4 ranks contend for the same PCIe link and the same host CPU cores (Colab gives 2 vCPUs for 4 Python processes).

**Why overlap lost here.**
1. *There is almost nothing to hide.* The deep-interior work per rank is only 254 × 254 ≈ 64.5 k cells (256 blocks). The kernel runs for tens of µs, and the halo messages are 2 048 B each. Open MPI moves these through shared memory (`vader` BTL) in a few µs. Total communication per step is 8 KB D2H + 8 KB H2D + 4 small messages.
2. *Overlap adds fixed costs.* Per step, serialized mode issues 3 kernels (pack, unpack, full stencil) + 2 copies + 2 stream syncs. Overlapped mode issues 7 kernels (pack, deep interior, unpack, 4 ring kernels) + 2 copies + 3 syncs across 2 streams. Every numba launch from Python costs tens of µs of host time, and with 4 processes time-slicing 2 vCPUs and one GPU, each extra launch and sync costs even more. The extra ≈ 1.08 ms/step (3.485 − 2.408) matches roughly 4 extra launches plus one extra stream synchronisation under contention.
3. *The ring kernels are inefficient by construction.* A 1 × 256 row launches 16 blocks of 256 threads, but only one row of each block (16 of 256 threads, 6 %) does work. The 254 × 1 columns have the same problem.
4. *Time-slicing removes the hardware concurrency that overlap relies on.* While rank A's interior kernel runs, rank B's copies and kernels wait for their slice, so the two streams of one rank cannot truly overlap with the other ranks' work.

**With 4 dedicated GPUs over InfiniBand** each rank owns all SMs and copy engines (no context switching). Inter-node communication then has a real cost: D2H + network latency of 1–2 µs plus bandwidth + H2D, i.e. tens of µs per exchange, or it can bypass host staging entirely with GPUDirect RDMA. At realistic sizes (e.g. 4096² cells per GPU) the interior kernel takes milliseconds, which is enough to hide the exchange completely. The step time then approaches max(t_interior, t_comm + t_ring) instead of their sum, and overlap gives a real speed-up (up to ~2× when t_comp ≈ t_comm). My result is the expected regime for a tiny problem oversubscribed on one shared GPU: launch and synchronisation overhead dominates, so the extra pipeline stages cost more than the communication they hide.

## Q5 — Point-to-point message deadlock & `MPI.PROC_NULL` semantics

**Eager vs rendezvous threshold.** A blocking `MPI_Send` returns once the send buffer can be reused, *not* when the message has been received. Below the transport's **eager limit**, the library copies the message into an internal buffer at the receiver (or in flight), and `Send` returns immediately even if no `Recv` is posted. Above the limit, Open MPI switches to the **rendezvous protocol**: the sender sends a small request-to-send and **blocks until the receiver posts the matching `Recv`** (clear-to-send). Then the payload moves. In Open MPI 4.1.6 (my run) the default eager limits are 4 KB for the shared-memory `vader` BTL, about 12 KB for `openib`, and 64 KB for TCP.

My halo messages are 256 × 8 B = **2 048 B**, below the 4 KB `vader` limit. So a naive blocking version would work *by accident* on this grid. On a larger grid, e.g. a 1024-cell edge = 8 KB (or 8192 cells = 64 KB over TCP), every `Send` becomes rendezvous. Now suppose every rank executes the same fixed order "`Send` up, `Send` down, `Send` left, `Send` right, then the four `Recv`s":
* rank 0 `(0,0)`: `Send` up → PROC_NULL (returns), then `Send` down → rank 2, **blocks** waiting for rank 2 to post `Recv`;
* rank 2 `(1,0)`: `Send` up → rank 0, **blocks** waiting for rank 0 to post `Recv`.

Each rank waits on a partner that is itself stuck in a `Send`. This is a circular wait (0 → 2 → 0, and likewise 1 ↔ 3 and the left/right pairs), so nobody ever reaches its `Recv` and the program hangs. Correctness then depends on the message size, which is why the bug typically shows up only after the grid is scaled up. With a "`Send` right then `Recv` left" order on a periodic grid, the cycle wraps through all P ranks. On a non-periodic grid the same pattern does not deadlock but serialises into a chain whose length grows with P.

My solver avoids both problems by construction. All four `Irecv` are **pre-posted** before any `Isend`, every call is non-blocking, and completion is a single `MPI.Request.Waitall(reqs)`. The library can match messages in any order, so there is no ordering dependency and no buffer-size dependency.

**`MPI.PROC_NULL` on corner ranks.** `cart.Shift(0,1)` / `cart.Shift(1,1)` on the non-periodic 2 × 2 grid return `PROC_NULL` beyond the domain edge. My run printed `rank 0 coords=(0, 0) up=PROC_NULL down=2 left=PROC_NULL right=1`. A send or receive whose peer is `PROC_NULL` **completes immediately and transfers no data**: the receive buffer is left untouched, with status source = `PROC_NULL` and count = 0. As a result, every rank executes exactly the same 4 `Irecv` + 4 `Isend` + `Waitall` code with no `if corner:` branches. Rank 0 actually exchanges only 2 messages (with rank 1 to the right and rank 2 below), and its other two requests are no-ops. On the GPU side `unpack_halos_kernel` receives `has_up/has_left = 0`, so ghost row 0 and ghost column 0 of rank 0 keep the value 0.0 set in `initialize_subdomains`. That value *is* the homogeneous Dirichlet boundary condition, so the boundary treatment needs no special stencil code either.

## Q6 — Explicit CFL stability & energy conservation/dissipation

**Stability limit from my run.**
* κ range (global `Allreduce` MIN/MAX): [0.080002, **0.239998**] → κ_max = 0.239998
* Δx = Δy = 1/513 = 1.9493177388 × 10⁻³ → Δx² = 3.7998 × 10⁻⁶
* Δt_max = Δx² / (4 κ_max) = 3.7998 × 10⁻⁶ / (4 × 0.239998) = **3.9581992407 × 10⁻⁶**
* Δt used = 0.8 · Δt_max = **3.1665593926 × 10⁻⁶** (printed `dt/dt_max = 0.800`)

Δt is 20 % below the 2-D von Neumann limit. Every interface conductivity is an arithmetic mean κ_{i±1/2,j} ≤ κ_max, so the bound holds for the variable-coefficient operator as well. Each update is then a convex combination of the 5 stencil values: the centre weight is 1 − Δt·(κ_e+κ_w+κ_n+κ_s)/Δx² ≥ 1 − 0.8 = 0.2 > 0. Hence no oscillation or growth can occur. The cubic term −λu³ with λ = 0.15 and |u| ≤ 0.92 adds at most λu²Δt ≈ 4 × 10⁻⁷ of extra damping per step, which is negligible for stability. The simulated physical time is 400 · Δt = 1.2666 × 10⁻³.

**Global `Allreduce` energy trace E = Σ u·Δx·Δy (identical in serialized and overlapped modes):**

| step | E | ΔE vs previous | L2 residual ‖uⁿ − uⁿ⁻¹‖₂ | max u |
|---|---|---|---|---|
| 0 | 6.220141891062554 × 10⁻² | — | 0 | 0.922413 |
| 100 | 6.212800344479495 × 10⁻² | −7.341547 × 10⁻⁵ | 1.583920 × 10⁻² | 0.885807 |
| 200 | 6.208424746570877 × 10⁻² | −4.375598 × 10⁻⁵ | 1.490631 × 10⁻² | 0.852387 |
| 300 | 6.204556092680388 × 10⁻² | −3.868654 × 10⁻⁵ | 1.411210 × 10⁻² | 0.822113 |
| 400 | 6.200914334756026 × 10⁻² | −3.641758 × 10⁻⁵ | 1.340204 × 10⁻² | 0.794480 |

**Interpretation.**
* E decreases **strictly monotonically** at every checkpoint (`energy monotonically non-increasing: True`). The total loss is 1.9228 × 10⁻⁴, i.e. 0.31 % over 400 steps.
* Energy budget: dE/dt = −(outflow through the Dirichlet walls) − λ Σ u³ ΔxΔy + Σ S ΔxΔy. The conservative flux form of the stencil makes all interior diffusive fluxes cancel in the sum (what leaves cell (i,j) enters its neighbour). Diffusion therefore only changes E through the boundary, where the ghost value u = 0 pulls heat out.
* Order-of-magnitude check with my seeded parameters: the cubic sink λ∫u³ ≈ 1.8 × 10⁻³ and the source ∫S ≈ 0.05·2π·0.03² ≈ 2.8 × 10⁻⁴. Their net effect is only ≈ −5 × 10⁻⁷ per 100 steps (100Δt = 3.17 × 10⁻⁴), while the measured drop is −7.3 × 10⁻⁵. The dominant loss is therefore the **Dirichlet boundary outflow**: the Gaussian tails are non-zero at the walls (e.g. the pulse at y = 0.80 with σ = 0.084 still has u ≈ 0.035 at y = 1), so a steep boundary layer forms between u > 0 and the ghost value 0.
* The loss rate shrinks from −7.34 × 10⁻⁵ to −3.64 × 10⁻⁵ per 100 steps. This is the classic diffusive boundary-layer behaviour: the initial jump at the wall gives a large flux, which decays roughly like t^(−1/2) as the layer thickens (√(κ_max t) ≈ 0.017 ≈ 9 cells at t = 1.27 × 10⁻³). It settles towards the slower bulk decay from cubic damping plus the constant source.
* The L2 increment norm decreases monotonically (1.584 × 10⁻² → 1.340 × 10⁻²) and the maximum principle holds (max u falls and never overshoots). These are the signatures of a stable, dissipative explicit scheme; an unstable one would show exponentially growing, oscillating increments.
* Both the serialized and the overlapped run reproduce this trace digit-for-digit, and their final fields agree with `parity_max_diff = 0.000e+00`. The asynchronous halo pipeline therefore changes neither stability nor the conservation behaviour.